<a href="https://colab.research.google.com/github/PauloRadatz/opendss-python-examples/blob/main/presentations/engr330_charleston_southern_university/overvoltage_distributed_fixed_locations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Distributed Overvoltage Hosting Capacity (Fixed Locations)

This notebook illustrates **distributed hosting capacity** with generation at fixed bus locations. Power is split equally among the generators. We compare two scenarios: generation at buses 4 and 7 vs. generation at buses 6 and 7.

In [1]:
!pip install py-dss-interface
!pip install py-dss-toolkit
!git clone https://github.com/PauloRadatz/opendss-python-examples.git
%cd opendss-python-examples

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 5.5 MB/s eta 0:00:00
Cloning into 'opendss-python-examples'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 73 (delta 19), reused 50 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 255.87 KiB | 10.23 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/opendss-python-examples


In [2]:
import pathlib
import numpy as np
import pandas as pd
import py_dss_interface
from py_dss_toolkit import dss_tools

# --- Helper functions ---
STEP_KW = 100
MAX_KW = 10000
OV_THRESHOLD = 1.05

def add_gen(dss, gen_bus, gen_kv):
    for gen in gen_bus:
        dss.text(f"new generator.{gen} phases=3 bus1={gen_bus[gen]} kv={gen_kv[gen]} "
                 f"kw=0.0001 pf=1 Vminpu=0.7 Vmaxpu=1.2")

def increase_gen(dss, gen_kw):
    for gen, kw in gen_kw.items():
        dss.text(f"Edit generator.{gen} kw={kw}")

def solve_powerflow(dss):
    dss.text("solve")

def check_overvoltage_violation(dss):
    voltages = dss.circuit.buses_vmag_pu
    return max(voltages) > OV_THRESHOLD

def set_load_level_condition(dss, load_mult):
    dss.text(f"set loadmult={load_mult}")

def result_distributed_info(buses, metric, hc_value, load_condition, existing_gen, device_type):
    return f"""Summary of the Distributed Hosting Capacity Analysis

Feeder Conditions:
    Load level condition = {load_condition}
    Existing Gen considered = {existing_gen}

Hosting Capacity:
    Buses = {buses}
    Value = {hc_value} kW
    Device Type = {device_type}
    Metric = {metric}"""

## Define feeder and create OpenDSS object

In [3]:
start_dir = pathlib.Path.cwd().resolve()
repo_root = next(
    parent for parent in [start_dir, *start_dir.parents]
    if (parent / "feeder_models").exists()
)
dss_file = repo_root / "feeder_models" / "8bus" / "Master.dss"
dss = py_dss_interface.DSS()

dss_tools.update_dss(dss)

print(f"Using feeder file: {dss_file}")

Using feeder file: /content/opendss-python-examples/feeder_models/8bus/Master.dss


## Scenario 1: Generation at buses 4 and 7

In [4]:
load_mult = 0.2
buses_1 = ["4", "7"]

dss.text(f'compile "{dss_file}"')
set_load_level_condition(dss, load_mult=load_mult)

gen_bus = {}
gen_kv = {}
for bus in buses_1:
    dss.circuit.set_active_bus(bus)
    kv = dss.bus.kv_base * np.sqrt(3)
    gen_bus[f"gen_{bus}"] = dss.bus.name
    gen_kv[f"gen_{bus}"] = kv

add_gen(dss, gen_bus, gen_kv)

hosting_capacity_1 = MAX_KW
i = 0
while i * STEP_KW < MAX_KW:
    i += 1
    i_kw = i * STEP_KW
    gen_kw = {gen: i_kw / len(gen_bus) for gen in gen_bus}

    increase_gen(dss, gen_kw)
    solve_powerflow(dss)

    if check_overvoltage_violation(dss):
        hosting_capacity_1 = (i - 1) * STEP_KW
        break

print(result_distributed_info(buses_1, "Overvoltage", hosting_capacity_1, "offpeak", False, "Generator"))
print(f"\nTotal HC: {hosting_capacity_1} kW ({hosting_capacity_1/1000:.2f} MW)")

Summary of the Distributed Hosting Capacity Analysis

Feeder Conditions:
    Load level condition = offpeak
    Existing Gen considered = False

Hosting Capacity:
    Buses = ['4', '7']
    Value = 1300 kW
    Device Type = Generator
    Metric = Overvoltage

Total HC: 1300 kW (1.30 MW)


## Scenario 2: Generation at buses 6 and 7

In [5]:
buses_2 = ["6", "7"]

dss.text(f'compile "{dss_file}"')
set_load_level_condition(dss, load_mult=load_mult)

gen_bus = {}
gen_kv = {}
for bus in buses_2:
    dss.circuit.set_active_bus(bus)
    kv = dss.bus.kv_base * np.sqrt(3)
    gen_bus[f"gen_{bus}"] = dss.bus.name
    gen_kv[f"gen_{bus}"] = kv

add_gen(dss, gen_bus, gen_kv)

hosting_capacity_2 = MAX_KW
i = 0
while i * STEP_KW < MAX_KW:
    i += 1
    i_kw = i * STEP_KW
    gen_kw = {gen: i_kw / len(gen_bus) for gen in gen_bus}

    increase_gen(dss, gen_kw)
    solve_powerflow(dss)

    if check_overvoltage_violation(dss):
        hosting_capacity_2 = (i - 1) * STEP_KW
        break

print(result_distributed_info(buses_2, "Overvoltage", hosting_capacity_2, "offpeak", False, "Generator"))
print(f"\nTotal HC: {hosting_capacity_2} kW ({hosting_capacity_2/1000:.2f} MW)")

Summary of the Distributed Hosting Capacity Analysis

Feeder Conditions:
    Load level condition = offpeak
    Existing Gen considered = False

Hosting Capacity:
    Buses = ['6', '7']
    Value = 1000 kW
    Device Type = Generator
    Metric = Overvoltage

Total HC: 1000 kW (1.00 MW)


## Comparison between scenarios

In [6]:
comparison_df = pd.DataFrame(
    {
        "Scenario": ["1: Buses 4 and 7", "2: Buses 6 and 7"],
        "Buses": [buses_1, buses_2],
        "Hosting Capacity (kW)": [hosting_capacity_1, hosting_capacity_2],
        "Hosting Capacity (MW)": [hosting_capacity_1/1000, hosting_capacity_2/1000],
    }
)
comparison_df

,Scenario,Buses,Hosting Capacity (kW),Hosting Capacity (MW)
0,1: Buses 4 and 7,"[4, 7]",1300,1.3
1,2: Buses 6 and 7,"[6, 7]",1000,1.0


In [7]:
diff_kw = hosting_capacity_1 - hosting_capacity_2
diff_pct = 100 * diff_kw / hosting_capacity_2 if hosting_capacity_2 > 0 else 0
print(f"Scenario 1 (buses 4, 7) has {diff_kw} kW ({diff_pct:.1f}%) more hosting capacity than Scenario 2 (buses 6, 7).")

Scenario 1 (buses 4, 7) has 300 kW (30.0%) more hosting capacity than Scenario 2 (buses 6, 7).


## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [pauloradatz.me](https://www.pauloradatz.me)
- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)